# HBC515 end-to-end SIGMA validation

This example runs the public `sigma-omics` API on HBC515 without importing code from the original notebook. It preserves the validated HBC515 model settings and writes all outputs to a new validation directory. The original data and notebooks are read-only inputs.

In [ ]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import sigma_spatial
from sigma_spatial import (run_sigma, anisotropy_report, plot_anisotropy_polar,
                           program_report_data, plot_program_report, score_st_signatures,
                           stroma_near_far_statistics, plot_stroma_near_far_validation)
from generate_hbc515_complete_report import GENE_SETS
from sigma_spatial.plotting import save_figure, set_publication_style, style_spatial_axis

set_publication_style()
SEED = 0
PROJECT_ROOT = Path.cwd().resolve().parents[1]
DATA_FILE = PROJECT_ROOT / 'HBC_515' / 'BC_515_Section_1.h5ad'
REFERENCE_FILE = PROJECT_ROOT / 'HBC_515' / 'clean' / 'results' / 'HBC515_SIGMA_core.h5ad'
OUTPUT_DIR = PROJECT_ROOT / 'HBC_515' / 'clean' / 'results' / 'pypi_validation_0.1.1'
FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print({'python': platform.python_version(), 'sigma_spatial': sigma_spatial.__version__, 'input': str(DATA_FILE)})

## 7. Optional matched-ST validation and automatic interface-program selection

For HBC515, matched RNA measurements are available. The frozen 5-spot near and 8-spot far definitions are applied to every metabolic program and RNA signature. The leading metabolic program must reproduce Cluster 3.

In [ ]:
signature_scores = score_st_signatures(result, GENE_SETS, random_state=SEED)
near_far_table, selected_program = stroma_near_far_statistics(
    result, program_report['scores'], signature_scores=signature_scores,
    spot_spacing=21.0, near_width=5, far_start=8,
)
near_far_table.to_csv(FULL_REPORT_DIR / 'HBC515_stroma_side_near_far_statistics.csv', index=False)
plot_stroma_near_far_validation(
    near_far_table, FULL_REPORT_DIR, leading_program=selected_program, prefix='HBC515'
)
assert selected_program == 3, f'Frozen HBC515 result expected Cluster 3, got {selected_program}'
print(f'Automatically selected boundary-associated program: Cluster {selected_program}')
display(near_far_table.sort_values('near_minus_far', ascending=False))

## 6. Generate the complete downstream validation report

This section uses the provenance-locked HBC515 program assignments. It does not recluster metabolites or change the original boundary-program interpretation. Cluster 3 is the frozen stromal-side boundary-associated program.

In [ ]:
FULL_REPORT_DIR = FIGURE_DIR / 'full_report'
assignments = pd.read_csv(PROJECT_ROOT / 'HBC_515' / 'clean' / 'results' / 'BC515_metabolic_program_assignments.csv')
program_report = program_report_data(result, assignments)
representatives = plot_program_report(
    result, program_report, FULL_REPORT_DIR, boundary_cluster=3, prefix='HBC515'
)
representatives.to_csv(FULL_REPORT_DIR / 'HBC515_boundary_program_representatives.csv', index=False)
display(representatives[['cluster', 'mz', 'lambda', 'r2_logI', 'boundary_enrichment_ratio']])

In [ ]:
mz_names = np.asarray(result.uns['mz_features'], dtype=float)
target_mz = 817.5345203639481
feature_index = int(np.argmin(np.abs(mz_names - target_mz)))
anisotropy = anisotropy_report(result, feature_index, n_sectors=8, min_points=200)
anisotropy['table'].to_csv(FULL_REPORT_DIR / 'HBC515_mz817_5345_sector_lambda.csv', index=False)
plot_anisotropy_polar(anisotropy, FULL_REPORT_DIR, prefix='HBC515_mz817_5345')
print(f"Anisotropy CV = {anisotropy['anisotropy_cv']:.3f}")
display(anisotropy['table'])

## 1. Load the public input and construct weak anchors

In [ ]:
adata = ad.read_h5ad(DATA_FILE)
annotation = adata.obs['annotation'].astype(str)
adata.obs['sigma_anchor'] = np.select(
    [annotation.eq('Tumor'), annotation.eq('Stroma')],
    [1.0, 0.0],
    default=np.nan,
)
print(adata)
print(adata.obs['sigma_anchor'].value_counts(dropna=False).sort_index())

## 2. Run the published SIGMA API

These are the frozen HBC515 reference settings: 64 MSI components, directed Gaussian kNN with k=15, 1,000 epochs, supervision weight 0.05, RNA alignment weight 0.05, and boundary k=10.

In [ ]:
result = run_sigma(
    adata,
    anchor_key='sigma_anchor',
    representation_key='X_harmony',
    spatial_key='spatial',
    copy=True,
    n_components=64,
    n_neighbors=15,
    random_state=SEED,
    z_dim=32,
    epochs=1000,
    lr=1e-3,
    lambda_sup=0.05,
    beta_rna=0.05,
    boundary_k=10,
)
result.write_h5ad(OUTPUT_DIR / 'HBC515_sigma_omics_0.1.1.h5ad')

## 3. Review and save the essential SIGMA fields

In [ ]:
xy = np.asarray(result.obsm['spatial'])
panels = [
    ('sigma_gaussian_anchor', 'Interface-aware prior', 'coolwarm'),
    ('sigma_region_probability', 'Tumor probability', 'viridis'),
    ('sigma_boundary', 'SIGMA boundary', 'Reds'),
    ('sigma_d_signed', 'Signed distance', 'coolwarm'),
]
fig, axes = plt.subplots(1, 4, figsize=(12.0, 3.0), constrained_layout=True)
for ax, (key, title, cmap) in zip(axes, panels):
    values = result.obs[key].to_numpy()
    artist = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=4, cmap=cmap, rasterized=True)
    ax.set_title(title)
    style_spatial_axis(ax)
    fig.colorbar(artist, ax=ax, shrink=0.72, pad=0.02)
save_figure(fig, FIGURE_DIR / 'HBC515_public_api_core_fields', formats=('pdf', 'svg', 'png'))
plt.show()

## 4. Numerical preservation check against the frozen reference

In [ ]:
reference = ad.read_h5ad(REFERENCE_FILE, backed='r')
checks = {}
for key in ['sigma_region_probability', 'sigma_d_signed']:
    observed = result.obs[key].to_numpy(dtype=float)
    expected = reference.obs[key].to_numpy(dtype=float)
    checks[key] = {
        'correlation': float(np.corrcoef(observed, expected)[0, 1]),
        'mae': float(np.mean(np.abs(observed - expected))),
    }
checks['sigma_boundary'] = {
    'agreement': float(np.mean(result.obs['sigma_boundary'].to_numpy() == reference.obs['sigma_boundary'].to_numpy()))
}
pd.DataFrame(checks).T.to_csv(OUTPUT_DIR / 'HBC515_public_api_regression.csv')
print(json.dumps(checks, indent=2))

## 5. Display the preserved HBC515 influence-range distribution

The current public API reproduces the SIGMA core. The provenance-locked HBC515 lambda table is loaded here for reporting; lambda estimation will become a public report component in a later release.

In [ ]:
lambda_table = pd.read_csv(PROJECT_ROOT / 'HBC_515' / 'clean' / 'results' / 'BC515_lambda_ranking.csv')
lambda_values = pd.to_numeric(lambda_table['lambda'], errors='coerce')
lambda_values = lambda_values[np.isfinite(lambda_values) & (lambda_values > 0)]
median_lambda = float(np.median(lambda_values))
fig, ax = plt.subplots(figsize=(5.0, 3.3), constrained_layout=True)
ax.hist(lambda_values, bins=35, color='#4C78A8', edgecolor='white', linewidth=0.4)
ax.axvline(median_lambda, color='#2457A7', linestyle='--', linewidth=1.4)
ax.text(median_lambda, ax.get_ylim()[1] * 0.90, f'Median λ = {median_lambda:.0f}', color='#2457A7', ha='left', va='top')
ax.set(xlabel='Interface influence range λ', ylabel='Count', title='HBC515 distribution of interface influence range')
ax.spines[['top', 'right']].set_visible(False)
save_figure(fig, FIGURE_DIR / 'HBC515_lambda_distribution', formats=('pdf', 'svg', 'png'))
plt.show()

## Outputs

The validation directory contains the new H5AD result, a regression table, core fields, influence range, anisotropy, all four frozen metabolic patterns, signed-distance profiles, the boundary-associated pattern, and representative metabolite spatial maps. ST validation is optional and remains separate when matched ST data are available; simulation, benchmark, GO enrichment and manuscript assembly are not part of this core validation run.